# Phase 4: Production ML Deployment & MLOps Infrastructure

Welcome to the final phase of the AuraCart MLOps initiative. In this notebook, we bridge the gap between offline experimentation and production inference. We will programmatically interact with Google Cloud Platform (GCP) to register our champion model, containerize it using Vertex AI's pre-built environments, and instantiate a live RESTful API endpoint.

As mandated by the AuraCart mandate, this system provides a unified, scalable interface for frontend microservices to query predictive logic in real-time using JSON payloads.

In [ ]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"c:\Users\ACER\Downloads\ML Test 4\notebooks\ml-final-project-492403-00f3e88c2f42.json"
import joblib
import pandas as pd
from google.cloud import storage
from google.cloud import aiplatform

# Configuration Variables (Production Parameters)
PROJECT_ID = "ml-final-project-492403"
REGION = "asia-southeast1"
BUCKET_NAME = f"{PROJECT_ID}-auracart-ml-artifacts-v2"
MODEL_NAME = "AuraCart_Segmenter_Champion-v2"
ENDPOINT_NAME = "auracart-prediction-endpoint-v2"
MODEL_ARTIFACT_PATH = "../artifacts/model.joblib"
REQUIREMENTS_FILE = "../artifacts/requirements.txt"

print(f"Deployment system initialized targeting Project: {PROJECT_ID} in {REGION}")


Deployment system initialized targeting Project: ml-final-project-492403 in asia-southeast1


### Environment Requirements Generation

To ensure the Vertex AI pre-built container has the correct dependencies (Scikit-Learn, Pandas, Imbalanced-Learn), we generate a `requirements.txt` file as an deployment artifact. This file will be uploaded to Cloud Storage alongside the model binary.

In [16]:
# Capture Dependencies
requirements_content = """
scikit-learn==1.8.0
pandas==2.3.3
numpy==2.2.6
joblib==1.5.2
imbalanced-learn==0.14.1
scipy==1.16.1
threadpoolctl==3.6.0
"""

REQUIREMENTS_FILE = "../artifacts/requirements.txt"
with open(REQUIREMENTS_FILE, "w") as f:
    f.write(requirements_content.strip())

print(f"Deployment requirements manifest generated at: {REQUIREMENTS_FILE}")

Deployment requirements manifest generated at: ../artifacts/requirements.txt


### Model Packaging & Artifact Management

As mandated by the Specification, **only one model will be deployed** to production: the final champion model trained for the **customer_segment prediction task**. We will serialize this model into a Scikit-learn Pipeline and prepare the dependency environmental manifest (`requirements.txt`) for cloud containerization.

In [22]:
print("Uploading Artifacts to Google Cloud Storage (GCS)...")
storage_client = storage.Client(project=PROJECT_ID)

bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Success: Bucket {BUCKET_NAME} created.")

# 1. Upload Model Artifact
model_blob = bucket.blob("model/model.joblib")
model_blob.upload_from_filename(MODEL_ARTIFACT_PATH, timeout=6000)
print(f"Artifact model.joblib uploaded to gs://{BUCKET_NAME}/model/")

# 2. Upload Requirements Artifact
req_blob = bucket.blob("model/requirements.txt")
req_blob.upload_from_filename(REQUIREMENTS_FILE, timeout=6000)
print(f"Artifact requirements.txt uploaded to gs://{BUCKET_NAME}/model/")

Uploading Artifacts to Google Cloud Storage (GCS)...
Artifact model.joblib uploaded to gs://ml-final-project-492403-auracart-ml-artifacts-v2/model/
Artifact requirements.txt uploaded to gs://ml-final-project-492403-auracart-ml-artifacts-v2/model/


### Deploying a Live Prediction Service with Vertex AI

We now ingest our model into the **Vertex AI Model Registry** and deploy it to a persistent **REST API Endpoint**. This provides a scalable, enterprise-grade inference service for AuraCart's web and mobile frontend applications.

In [23]:
print("Initializing Vertex AI SDK...")
aiplatform.init(project=PROJECT_ID, location=REGION)

# 1. Upload Model to Registry
print("Registering model in Vertex AI Registry...")
model_registered = aiplatform.Model.upload(
    display_name=MODEL_NAME,
    artifact_uri=f"gs://{BUCKET_NAME}/model/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest",
)


Initializing Vertex AI SDK...
Registering model in Vertex AI Registry...
Creating Model
Create Model backing LRO: projects/38440670392/locations/asia-southeast1/models/5409751140285808640/operations/3889055365693702144
Model created. Resource name: projects/38440670392/locations/asia-southeast1/models/5409751140285808640@1
To use this Model in another session:
model = aiplatform.Model('projects/38440670392/locations/asia-southeast1/models/5409751140285808640@1')


In [24]:
# 2. Create Endpoint explicitly
print(f"Creating Endpoint: {ENDPOINT_NAME}...")
endpoint = aiplatform.Endpoint.create(display_name=ENDPOINT_NAME)



Creating Endpoint: auracart-prediction-endpoint-v2...
Creating Endpoint
Create Endpoint backing LRO: projects/38440670392/locations/asia-southeast1/endpoints/6383819486448844800/operations/7057337703548846080
Endpoint created. Resource name: projects/38440670392/locations/asia-southeast1/endpoints/6383819486448844800
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/38440670392/locations/asia-southeast1/endpoints/6383819486448844800')


In [25]:
# 3. Deploy model to Endpoint
print("Deploying model to endpoint (this may take several minutes)... ")
model_registered.deploy(
    endpoint=endpoint,
    machine_type="n1-standard-2",
)

print(f"Successfully deployed AuraCart Prediction Engine to Endpoint ID: {endpoint.resource_name}")

Deploying model to endpoint (this may take several minutes)... 
Deploying model to Endpoint : projects/38440670392/locations/asia-southeast1/endpoints/6383819486448844800
Deploy Endpoint model backing LRO: projects/38440670392/locations/asia-southeast1/endpoints/6383819486448844800/operations/6363361148468723712
Endpoint model deployed. Resource name: projects/38440670392/locations/asia-southeast1/endpoints/6383819486448844800
Successfully deployed AuraCart Prediction Engine to Endpoint ID: projects/38440670392/locations/asia-southeast1/endpoints/6383819486448844800


### Deploying a Live Prediction Service with Vertex AI

We now ingest our model into the **Vertex AI Model Registry** and deploy it to a persistent **REST API Endpoint**. This provides a scalable, enterprise-grade inference service for AuraCart's web and mobile frontend applications.

In [26]:
# Live Test instance with 22 features in the correct training order
test_instance = [
    "Home & Kitchen", # category
    155.50,           # price
    2,                # quantity
    "Delivered",      # delivery_status
    "Credit Card",    # payment_method
    "Mobile",         # device_type
    "Social Media",   # channel
    3.0,              # shipping_delay_days
    4,                # order_month
    12,               # order_day
    6,                # order_dayofweek
    14,               # order_hour (Added)
    4,                # shipping_month
    15,               # shipping_day
    2,                # shipping_dayofweek
    14,               # shipping_hour (Added)
    1,                # is_weekend_order
    "San Jose",       # shipping_city
    "CA",             # shipping_state
    "San Jose",       # billing_city
    "CA",             # billing_state
    85                # product_popularity
]
print("Querying Vertex AI REST API with corrected schema...")
prediction = endpoint.predict(instances=[test_instance])
print(f"\nPrediction Response: {prediction.predictions[0]}")

Querying Vertex AI REST API with corrected schema...


InternalServerError: 500 {"detail":"The following exception has occurred: NotFittedError. Arguments: (\"This ColumnTransformer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.\",)."}